# LabCleaning_160004613
Se carga el dataset `dirty_cafe_sales.csv` y se aplica limpieza de datos para faltantes, valores atípicos, verificación de duplicados e imputaciones solicitadas.

In [ ]:
import numpy as np
import pandas as pd

ruta = 'dirty_cafe_sales.csv'
df_raw = pd.read_csv(ruta)
df_raw.head()

Técnica 1 de faltantes: conteo absoluto de valores nulos por columna.

In [ ]:
df_tmp = df_raw.replace(['ERROR', 'UNKNOWN', 'None'], np.nan)
df_tmp.isna().sum().sort_values(ascending=False)

Técnica 2 de faltantes: porcentaje de valores nulos por columna.

In [ ]:
(df_tmp.isna().mean() * 100).sort_values(ascending=False).round(2)

In [ ]:
df = df_raw.replace(['ERROR', 'UNKNOWN', 'None'], np.nan).copy()

for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])
df['Quantity'] = df['Quantity'].fillna(df['Total Spent'] / df['Price Per Unit'])
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Total Spent'] / df['Quantity'])

precios_por_item = df.groupby('Item', dropna=False)['Price Per Unit'].nunique(dropna=True).sort_values(ascending=False)
precios_por_item

In [ ]:
duplicados_txn = df['Transaction ID'].duplicated().sum()
duplicados_fila = df.duplicated().sum()
columnas_irrelevantes = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
pd.Series({
    'Duplicados Transaction ID': duplicados_txn,
    'Duplicados de fila completa': duplicados_fila,
    'Columnas irrelevantes (nunique<=1)': str(columnas_irrelevantes)
})

In [ ]:
media_total = df['Total Spent'].mean()
std_total = df['Total Spent'].std()
limite_inf = media_total - 3 * std_total
limite_sup = media_total + 3 * std_total
outliers_total_spent = df[(df['Total Spent'] < limite_inf) | (df['Total Spent'] > limite_sup)]
pd.Series({
    'Media Total Spent': round(media_total, 4),
    'Desviación estándar': round(std_total, 4),
    'Límite inferior': round(limite_inf, 4),
    'Límite superior': round(limite_sup, 4),
    'Cantidad outliers': len(outliers_total_spent)
})

In [ ]:
df_central = df.copy()
for col in ['Payment Method', 'Location']:
    moda_col = df_central[col].mode(dropna=True)
    moda = moda_col.iloc[0] if not moda_col.empty else np.nan
    df_central[col] = df_central[col].fillna(moda)

df_random = df.copy()
rng = np.random.default_rng(42)
for col in ['Payment Method', 'Location']:
    mascara = df_random[col].isna()
    valores = df_random.loc[~mascara, col].to_numpy()
    if mascara.sum() > 0 and len(valores) > 0:
        df_random.loc[mascara, col] = rng.choice(valores, size=mascara.sum(), replace=True)

df_central[['Payment Method', 'Location']].isna().sum().rename('Nulos tras imputación central'), df_random[['Payment Method', 'Location']].isna().sum().rename('Nulos tras imputación aleatoria')

In [ ]:
df_final = df_central.copy()
df_final['Transaction Date'] = pd.to_datetime(df_final['Transaction Date'], errors='coerce')
date_num = df_final['Transaction Date'].astype('int64').where(df_final['Transaction Date'].notna(), np.nan)
date_num = date_num.interpolate(method='linear', limit_direction='both')
df_final['Transaction Date'] = pd.to_datetime(date_num)

df_final[['Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']].isna().sum()

In [ ]:
df_final.head()